In [1]:
import re
from pathlib import Path

import pandas as pd

In [2]:
paths = sorted(Path("output/input_space_v2").rglob("eval_table.csv"))
print(len(paths))

tables = []
for path in paths:
    table = pd.read_csv(path)
    key = path.parts[-4]
    match = re.search(r"(schaefer400|mni_cortex|flat)_lr([-0-9e]+)_([0-9])", key)
    space, lr, trial = match.groups()
    table.insert(0, "trial", int(trial))
    table.insert(0, "base_lr", float(lr))
    table.insert(0, "space", space)
    table.insert(0, "key", key)
    tables.append(table)
table = pd.concat(tables, ignore_index=True)
print(table.shape)
table.head()

84
(2982, 21)


,key,space,base_lr,trial,model,repr,clf,dataset,trial_id,C,...,acc,epoch,lr,wd,hparam_id,hparam,loss,acc_std,f1,f1_std
0,mni_cortex_lr1e-3_1,mni_cortex,0.001,1,mni_cortex_mae,cls,logistic,aabc_age,0.0,0.005995,...,0.547619,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mni_cortex_lr1e-3_1,mni_cortex,0.001,1,mni_cortex_mae,cls,logistic,aabc_age,1.0,0.005995,...,0.583333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,mni_cortex_lr1e-3_1,mni_cortex,0.001,1,mni_cortex_mae,cls,logistic,aabc_age,2.0,0.005995,...,0.505952,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,mni_cortex_lr1e-3_1,mni_cortex,0.001,1,mni_cortex_mae,cls,logistic,aabc_age,3.0,0.005995,...,0.523810,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mni_cortex_lr1e-3_1,mni_cortex,0.001,1,mni_cortex_mae,cls,logistic,aabc_age,4.0,0.046416,...,0.613095,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
summary = table.loc[table["split"] == "test"].pivot_table(
    values=["acc", "acc_std"], index=["space", "base_lr", "trial", "repr", "clf"], columns="dataset"
)
summary = summary.loc[(slice(None), slice(None), slice(None), "patch", "attn")].dropna(
    axis=1, how="all"
)
summary

acc                                    \
dataset                    aabc_age hcpya_rest1lr_gender hcpya_task21   
space       base_lr trial                                               
mni_cortex  0.00030 1      0.528302             0.865385     0.963690   
                    2      0.452830             0.865385     0.957937   
            0.00100 1      0.528302             0.894231     0.963690   
                    2      0.490566             0.855769     0.961310   
schaefer400 0.00003 1      0.423077             0.596154     0.962897   
                    2      0.442308             0.625000     0.960119   
            0.00010 1      0.346154             0.663462     0.968452   
                    2      0.307692             0.711538     0.975992   
            0.00030 1      0.403846             0.730769     0.979960   
                    2      0.288462             0.759615     0.976786   
            0.00100 1      0.384615             0.740385     0.976190   
                    2      0.423077             0.769231     0.976190   
            0.00300 1      0.326923             0.692308     0.980952   
                    2      0.326923             0.759615     0.980159   

                                         acc_std                       \
dataset                   nsd_cococlip  aabc_age hcpya_rest1lr_gender   
space       base_lr trial                                               
mni_cortex  0.00030 1         0.260482  0.067958             0.033375   
                    2         0.272356  0.060530             0.033023   
            0.00100 1         0.263636  0.061110             0.030447   
                    2         0.253247  0.063688             0.035799   
schaefer400 0.00003 1         0.256030  0.056038             0.046469   
                    2         0.264750  0.061480             0.046939   
            0.00010 1         0.272542  0.067755             0.050960   
                    2         0.286456  0.062527             0.044628   
            0.00030 1         0.278664  0.063706             0.043134   
                    2         0.279221  0.057633             0.041263   
            0.00100 1         0.271243  0.064464             0.043340   
                    2         0.269017  0.059915             0.042876   
            0.00300 1         0.264564  0.061140             0.044607   
                    2         0.266605  0.057814             0.042892   

                                                     
dataset                   hcpya_task21 nsd_cococlip  
space       base_lr trial                            
mni_cortex  0.00030 1         0.002627     0.005052  
                    2         0.002718     0.005273  
            0.00100 1         0.002561     0.005004  
                    2         0.002701     0.004835  
schaefer400 0.00003 1         0.002510     0.005119  
                    2         0.002771     0.005016  
            0.00010 1         0.002446     0.005045  
                    2         0.002181     0.005165  
            0.00030 1         0.001961     0.005149  
                    2         0.002093     0.005020  
            0.00100 1         0.002072     0.005371  
                    2         0.002158     0.005094  
            0.00300 1         0.001895     0.005381  
                    2         0.002065     0.005110

In [4]:
DATASET_NAMES = {
    "abide_dx": "ABIDE Dx",
    "abide_age": "ABIDE Age",
    "abide_sex": "ABIDE Sex",
    "adhd200_dx": "ADHD200 Dx",
    "adhd200_sex": "ADHD200 Sex",
    "adni_ad_vs_cn": "ADNI Dx",
    "adni_sex": "ADNI Sex",
    "ppmi_dx": "PPMI Dx",
    "ppmi_sex": "PPMI Sex",
    "ppmi_age": "PPMI Age",
    "hcpya_rest1lr_gender": "HCP-YA Sex",
    "hcpya_rest1lr_age": "HCP-YA Age",
    "hcpya_rest1lr_neofacn": "HCP-YA NEO-N",
    "aabc_sex": "HCP-A Sex",
    "aabc_age": "HCP-A Age",
    "hcpya_task21": "HCP-YA Task21",
    "nsd_cococlip": "NSD COCO24",
}

SPACE_NAMES = {
    "schaefer400": "parcel",
    "flat": "flat",
    "mni_cortex": "volume",
}


def format_acc_std(acc, std):
    """Format accuracy and std as 'XX.XX ± YY.YY'"""
    if pd.isna(acc) or pd.isna(std):
        return "—"
    return f"{acc * 100:.1f} ± {std * 100:.1f}"
    # return f"{acc * 100:.1f} \mypm{{{std * 100:.1f}}}"

In [5]:
# datasets = ["hcpya_rest1lr_gender", "aabc_sex", "hcpya_task21", "nsd_cococlip"]
datasets = ["aabc_age", "hcpya_rest1lr_gender", "hcpya_task21", "nsd_cococlip"]

repr_, clf = "patch", "attn"

records = []

for (space, base_lr, trial), row in summary.iterrows():
    record = {"space": space, "lr": base_lr, "trial": trial}
    for ds in datasets:
        acc = row[("acc", ds)]
        std = row[("acc_std", ds)]
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "$\mypm$"))

| space       |     lr |   trial | HCP-A Age   | HCP-YA Sex   | HCP-YA Task21   | NSD COCO24   |
|:------------|-------:|--------:|:------------|:-------------|:----------------|:-------------|
| mni_cortex  | 0.0003 |       1 | 52.8 ± 6.8  | 86.5 ± 3.3   | 96.4 ± 0.3      | 26.0 ± 0.5   |
| mni_cortex  | 0.0003 |       2 | 45.3 ± 6.1  | 86.5 ± 3.3   | 95.8 ± 0.3      | 27.2 ± 0.5   |
| mni_cortex  | 0.001  |       1 | 52.8 ± 6.1  | 89.4 ± 3.0   | 96.4 ± 0.3      | 26.4 ± 0.5   |
| mni_cortex  | 0.001  |       2 | 49.1 ± 6.4  | 85.6 ± 3.6   | 96.1 ± 0.3      | 25.3 ± 0.5   |
| schaefer400 | 3e-05  |       1 | 42.3 ± 5.6  | 59.6 ± 4.6   | 96.3 ± 0.3      | 25.6 ± 0.5   |
| schaefer400 | 3e-05  |       2 | 44.2 ± 6.1  | 62.5 ± 4.7   | 96.0 ± 0.3      | 26.5 ± 0.5   |
| schaefer400 | 0.0001 |       1 | 34.6 ± 6.8  | 66.3 ± 5.1   | 96.8 ± 0.2      | 27.3 ± 0.5   |
| schaefer400 | 0.0001 |       2 | 30.8 ± 6.3  | 71.2 ± 4.5   | 97.6 ± 0.2      | 28.6 ± 0.5   |
| schaefer400 | 0.0003 |      

In [6]:
def sem(x):
    return x.std() / (len(x) ** 0.5)


logistic_summary = table.loc[(table["split"] == "test") & (table["clf"] == "logistic")].pivot_table(
    values=["acc"],
    index=["space", "base_lr", "trial", "repr", "clf"],
    columns="dataset",
    aggfunc=["mean", sem],
)
logistic_summary = logistic_summary.loc[
    (slice(None), slice(None), slice(None), "cls", "logistic")
].dropna(axis=1, how="all")
logistic_summary

mean                                 \
                                acc                                  
dataset                    aabc_age  aabc_sex  abide_dx adhd200_dx   
space       base_lr trial                                            
mni_cortex  0.0010  1      0.522202  0.855000  0.593790   0.603488   
                    2      0.535417  0.852273  0.576129   0.593566   
schaefer400 0.0003  1      0.433036  0.711250  0.609194   0.608682   
                    2      0.438333  0.691080  0.617460   0.590698   

                                                                        \
                                                                         
dataset                   adni_ad_vs_cn hcpya_rest1lr_gender   ppmi_dx   
space       base_lr trial                                                
mni_cortex  0.0010  1          0.771951             0.908889  0.630000   
                    2          0.769675             0.901111  0.629698   
schaefer400 0.0003  1          0.762764             0.800000  0.629648   
                    2          0.762276             0.794333  0.642161   

                                sem                                 \
                                acc                                  
dataset                    aabc_age  aabc_sex  abide_dx adhd200_dx   
space       base_lr trial                                            
mni_cortex  0.0010  1      0.003397  0.002482  0.002529   0.003611   
                    2      0.003296  0.002469  0.002622   0.003633   
schaefer400 0.0003  1      0.003477  0.003389  0.002366   0.003429   
                    2      0.002789  0.002970  0.002814   0.003562   

                                                                        
                                                                        
dataset                   adni_ad_vs_cn hcpya_rest1lr_gender   ppmi_dx  
space       base_lr trial                                               
mni_cortex  0.0010  1          0.002378             0.001764  0.002224  
                    2          0.002489             0.001816  0.002392  
schaefer400 0.0003  1          0.001868             0.002636  0.002714  
                    2          0.001738             0.002449  0.002617